# Example notebook for registering a new circuit with remote validation
- Checks metadata provided
- Registers circuit entity as `draft` and uploads the SONATA circuit folder directly to entitycore
- Starts validation on the obi-one service, which generates the assets after successful validation

## Circuit information

### General metadata

To be provided as a dict with key/value pairs. Optional entries can be set to None.

**IMPORTANT:**
- **The circuit name must not yet exist in entitycore (to avoid duplicates)**

- **Linked entities must exist in entitycore. Otherwise, they must first be registered:**
  - Species
  - Subject
  - Brain region hierarchy
  - Brain region
  - License


In [1]:
from entitysdk.types import CircuitBuildCategory, TargetSimulator

In [ ]:
circuit_metadata = {
    # Name of the circuit
    "name": "EXAMPLE_01__N_10__top_nodes_dim6__remote_validation",

    # Description of the circuit
    "description": "Example circuit with 10 neurons",
    
    # Circuit build category (computational_model, em_reconstruction)
    "build_category": CircuitBuildCategory.computational_model,

    # Species name [Must exist in entitycore; otherwise, must first be registered]
    "species": "Rattus norvegicus",

    # Subject name [Must exist in entitycore; otherwise, must first be registered]
    "subject": "Average rat",

    # Brain region hierarchy name for the given species [Must exist in entitycore; otherwise, must first be registered]
    "brain_region_hierarchy": "Waxholm Space atlas of the rat brain with layers and hippocampal subregions",

    # Brain region name within hierarchy [Must exist in entitycore; otherwise, must first be registered]
    "brain_region": "Primary somatosensory area, hindlimb representation",

    # Target simulator (NEURON, Brian2, ...) [Must be consistent with circuit config if specified there]
    "target_simulator": TargetSimulator.NEURON,

    # Optional: Name of the root circuit (= top parent in the extraction/derivation hierarchy), if any
    "root": None,

    # Optional: Name of the parent circuit (=direct parent the circuit was extracted or derived from)
    "parent": None,

    # Optional: Type of circuit derivation (circuit_extraction, circuit_rewiring), if parent provided
    "derivation_type": None,

    # Optional: License name [Must exist in entitycore; otherwise, must first be registered]
    "license": "CC BY 4.0",

    # Optional: Short, human readable publication string
    "published_in": "Reimann et al and Isbister et al",

    # Optional: Contact e-mail address
    "contact": "support@openbraininstitute.org",

    # Optional: Experiment date (=date when the circuit has been build or published)
    # Note: Two date formats supported: "November, 2024" (treated as 01.11.2024) and "27.03.2024"
    "experiment_date": "15.05.2025",
    }

### Circuit path

Path to the SONATA circuit, which can be one of:
- A folder containing the `circuit_config.json`
- A direct path to the `circuit_config.json` file
- A compressed `.gz` archive of the circuit folder

In [3]:
# SONATA circuit folder with circuit_config.json (or path to .gz archive)
circuit_path = "../../data/tiny_circuits/N_10__top_nodes_dim6"

# Optional: Pre-existing image files (skip generation if provided)
overview_image_path = None  # Path to overview image (.png or .webp)
sim_designer_image_path = None  # Path to simulation designer image (.png)

### Optional: Contributors

To be provided as a dict:
- Key = Agent name **[Must exist in entitycore; otherwise, must first be registered]**
- Value = Dict with
  - "type" (person, organization, consortium)
  - "role" (unspecified, ...)

In [ ]:
circuit_contributions = {
    # Contributing persons
    "Christoph Pokorny": {"type": "person", "role": "unspecified"},
    # "<ADD NAME>": {"type": "person", "role": "unspecified"},

    # Contributing organizations
    "Open Brain Institute, Lausanne, Switzerland": {"type": "organization", "role": "unspecified"},
    # "<ADD NAME>": {"type": "organization", "role": "unspecified"},

    # Contributing consortia
    # "<ADD NAME>": {"type": "consortium", "role": "unspecified"},
 }

### Optional: Publications

To be provided as a dict:
- Key = DOI **[Must exist in entitycore; otherwise, must first be registered]**
- Value = Dict with
  - "type" (entity_source, component_source, application)

In [ ]:
circuit_publications = {
    # Circuit source publication(s)
    "10.7554/eLife.99688.2": {"type": "entity_source"},
    "10.7554/eLife.99693.2": {"type": "entity_source"},

    # Component source publication(s)
    # "<ADD DOI>": {"type": "component_source"},

    # Circuit application(s)
    # "<ADD DOI>": {"type": "application"},
    }

## Get DB client and obi-one client

Specify environment (staging, production) and project context.

- `client` (entitysdk) is used to upload the circuit files directly to entitycore.
- `obi_one_client` (httpx) is only used to start the validation on the obi-one service. Its `base_url` must include the `/api/obi-one` prefix. The authorization header is fixed when the client is created, so create it right before the registration if the token is short-lived.

In [6]:
import httpx
from entitysdk import Client, ProjectContext, models
from obi_auth import get_token

from obi_one import LAB_ID_STAGING_TEST, PROJECT_ID_STAGING_TEST

In [7]:
environment = "staging"
virtual_lab_id = LAB_ID_STAGING_TEST
project_id = PROJECT_ID_STAGING_TEST

# Get DB client
token = get_token(environment=environment)
project_context = ProjectContext(virtual_lab_id=virtual_lab_id, project_id=project_id)
client = Client(environment=environment, project_context=project_context, token_manager=token)

# Get obi-one API client (the project headers are added per request by the registration function)
OBI_ONE_API_URL = "https://staging.cell-a.openbraininstitute.org/api/obi-one"  # Deployed service
# OBI_ONE_API_URL = "http://127.0.0.1:8100"  # Local service (make run-local)
obi_one_client = httpx.Client(base_url=OBI_ONE_API_URL, headers={"Authorization": f"Bearer {token}"}, timeout=60.0)


## Resolve metadata

Resolves all linked entities from the metadata and checks that the circuit name does not yet exist.

In [8]:
from obi_one.db_sdk.registration.circuit.resolve import (
    check_hierarchy_species,
    check_if_circuit_exists,
    get_brain_region,
    get_brain_region_hierarchy,
    get_contributions,
    get_exp_date,
    get_license,
    get_parent_circuit,
    get_publications,
    get_root_circuit,
    get_subject,
)

In [9]:
check_if_circuit_exists(client, circuit_metadata)

subject = get_subject(client, circuit_metadata)
brain_hierarchy = get_brain_region_hierarchy(client, circuit_metadata)
check_hierarchy_species(brain_hierarchy, subject)
brain_region = get_brain_region(client, circuit_metadata, brain_hierarchy)
license_entity = get_license(client, circuit_metadata)
root = get_root_circuit(client, circuit_metadata)
parent = get_parent_circuit(client, circuit_metadata)
experiment_date = get_exp_date(circuit_metadata)

contributions = get_contributions(client, circuit_contributions) if circuit_contributions else None
publications = get_publications(client, circuit_publications) if circuit_publications else None

Person 'Christoph Pokorny' found multiple times - using first instance


## Run circuit registration

First, a dry run (`dry_run = True`) should be used to check the circuit files, metadata, and dependencies but w/o registration to entitycore and w/o contacting the obi-one service:
- The circuit folder must contain a loadable `circuit_config.json` and a `node_sets.json`, and the circuit size and properties must be computable from the node/edge files
- The target simulator must be consistent with the circuit config
- The circuit name must not yet exist in entitycore (i.e., to avoid duplicates)
- All dependencies must already exist in entitycore (i.e., must be registered beforehand)

Note that the full SONATA validation (including MOD/HOC checks) is **not** part of the dry run. It runs remotely after the actual registration and can still disqualify the circuit.

If successful, the actual registration can be done by re-running with `dry_run = False`. The circuit is then registered as `draft`, and the validation job is started on the obi-one service. After successful validation, the circuit becomes `active` and its additional assets are generated by a follow-up job.

The circuit can be made public by setting `authorized_public = True`, otherwise it will be private within the chosen project.

In [ ]:
from obi_one.db_sdk.registration.circuit.remote import register_circuit_with_remote_validation

In [19]:
dry_run = False  # If True, runs all checks but no actual registration; otherwise actual registration
authorized_public = False  # If True, registered as public entity; otherwise private within project

In [20]:
# Register circuit as draft & start remote validation
registered_circuit, job_id = register_circuit_with_remote_validation(
    client=client,
    circuit_path=circuit_path,
    obi_one_client=obi_one_client,
    name=circuit_metadata["name"],
    description=circuit_metadata["description"],
    build_category=circuit_metadata["build_category"],
    brain_region=brain_region,
    subject=subject,
    target_simulator=circuit_metadata["target_simulator"],
    contact_email=circuit_metadata.get("contact"),
    published_in=circuit_metadata.get("published_in"),
    experiment_date=experiment_date,
    license=license_entity,
    root=root,
    parent=parent,
    derivation_type=circuit_metadata.get("derivation_type"),
    contributions=contributions,
    publications=publications,
    authorized_public=authorized_public,
    overview_image_path=overview_image_path,
    sim_designer_image_path=sim_designer_image_path,
    dry_run=dry_run,
)
print(f"Registered circuit ID: {registered_circuit.id}" + (" (DRY RUN)" if dry_run else ""))
print(f"Validation job ID: {job_id}" + (" (DRY RUN)" if dry_run else ""))

1 '.DS_Store' file(s) found in '../../data/tiny_circuits/N_10__top_nodes_dim6' - ignoring


Registered circuit ID: eb4ba6cd-ce76-4175-8321-a1235f348d5c
Validation job ID: 01a1b1b7-8816-7740-b3e8-f131dd87060a


## Follow validation

The job can also be followed via `GET /declared/task/{job_id}` and `GET /declared/task/{job_id}/stream` on the obi-one service. Here, the lifecycle status of the circuit is polled until it leaves `draft` (`active` if validation passed, `disqualified` if it failed).

In [21]:
import time

if registered_circuit and registered_circuit.id:
    timeout_s, poll_s = 1800, 15
    start = time.time()
    while True:
        res = client.get_entity(entity_id=registered_circuit.id, entity_type=models.Circuit)
        print(f"[{int(time.time() - start):>5d} s] lifecycle_status={res.lifecycle_status}")
        if str(res.lifecycle_status) != "draft" or time.time() - start > timeout_s:
            break
        time.sleep(poll_s)
else:
    print("Circuit not registered!")

[    0 s] lifecycle_status=draft
[   16 s] lifecycle_status=draft
[   31 s] lifecycle_status=draft
[   47 s] lifecycle_status=draft
[   63 s] lifecycle_status=draft
[   78 s] lifecycle_status=draft
[   94 s] lifecycle_status=draft
[  110 s] lifecycle_status=draft
[  126 s] lifecycle_status=active


After validation, the Circuit entity can be checked. The additional assets (compressed circuit, connectivity matrices, plots, images) are generated by a follow-up job after successful validation, so they may appear a bit later.

In [22]:
if registered_circuit and registered_circuit.id:
    # Fetch entity incl. assets
    res = client.get_entity(entity_id=registered_circuit.id, entity_type=models.Circuit)
    
    # Print circuit info
    print(f"Circuit '{res.name}' registered (ID {res.id}), lifecycle_status={res.lifecycle_status}")
    print(f"  scale={res.scale}, neurons={res.number_neurons}, synapses={res.number_synapses}, connections={res.number_connections}")
    print(f"  has_morphologies={res.has_morphologies}, has_point_neurons={res.has_point_neurons}, has_electrical_cell_models={res.has_electrical_cell_models}, has_spines={res.has_spines}")

    # Print asset info
    print(f"Assets registered ({len(res.assets)}):")
    for a in res.assets:
        print(f"  Asset '{a.label}' ({'dir' if a.is_directory else 'file'}) registered (ID {a.id})")
else:
    print("Circuit not registered!")

Circuit 'EXAMPLE_01__N_10__top_nodes_dim6__remote_validation' registered (ID eb4ba6cd-ce76-4175-8321-a1235f348d5c), lifecycle_status=active
  scale=small, neurons=10, synapses=176, connections=23
  has_morphologies=True, has_point_neurons=False, has_electrical_cell_models=True, has_spines=False
Assets registered (1):
  Asset 'sonata_circuit' (dir) registered (ID 5db34bf7-5545-4573-a9cf-6f753c333765)
